# CarWorth — SHAP Explainability
**Goal:** Understand which features drive predictions, globally and per-car.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import json
import warnings

warnings.filterwarnings('ignore')
shap.initjs()

## 1. Load Model & Data

In [ ]:
model    = joblib.load('../models/xgb_model.joblib')
encoders = joblib.load('../models/encoders.joblib')

with open('../models/feature_names.json') as f:
    FEATURES = json.load(f)

df = pd.read_csv('../data/processed/vehicles_clean.csv')

# Re-encode categoricals (same as training)
CAT_COLS = ['manufacturer', 'fuel', 'transmission', 'drive', 'type', 'cylinders', 'state']
for col in [c for c in CAT_COLS if c in df.columns]:
    df[col] = encoders[col].transform(df[col].astype(str))

X = df[FEATURES]
print(f'X shape: {X.shape}')

## 2. Compute SHAP Values

In [ ]:
# Sample for speed (SHAP is O(n))
X_sample = X.sample(n=min(3000, len(X)), random_state=42)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

print(f'SHAP values shape: {shap_values.shape}')

## 3. Global Feature Importance — Beeswarm Plot

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, show=False)
plt.title('SHAP Summary — Feature Impact on log(Price)')
plt.tight_layout()
plt.savefig('../assets/04_shap/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Mean |SHAP| Bar Chart

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, plot_type='bar', show=False)
plt.title('Mean |SHAP| — Global Feature Importance')
plt.tight_layout()
plt.savefig('../assets/04_shap/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SHAP Dependence Plots — Top Features

In [ ]:
top_features = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURES
).sort_values(ascending=False).head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    shap.dependence_plot(
        feat, shap_values, X_sample,
        feature_names=FEATURES,
        ax=axes[i], show=False
    )
    axes[i].set_title(f'SHAP Dependence: {feat}')

plt.tight_layout()
plt.savefig('../assets/04_shap/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Single Prediction Explanation — Waterfall Plot

In [ ]:
# Explain one sample car
idx = 0
sample_car = X_sample.iloc[[idx]]
actual_price = np.expm1(df.loc[X_sample.index[idx], 'log_price'])
predicted_price = np.expm1(model.predict(sample_car)[0])

print(f'Actual Price   : ${actual_price:,.0f}')
print(f'Predicted Price: ${predicted_price:,.0f}')

explanation = shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=sample_car.values[0],
    feature_names=FEATURES
)

plt.figure(figsize=(10, 5))
shap.waterfall_plot(explanation, show=False)
plt.title(f'Why did CarWorth predict ${predicted_price:,.0f}?')
plt.tight_layout()
plt.savefig('../assets/04_shap/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save SHAP Explainer for Streamlit

In [ ]:
joblib.dump(explainer, '../models/shap_explainer.joblib')
print('Saved: models/shap_explainer.joblib')